# Day 13 · Lab 2/4: dates, aggregations, pivots and arrays, and how Spark plans them

Lab 1 taught the DataFrame API that Zillacode tests. From today, every lab has two tracks:
* ✍️ **Write it**: the DataFrame API for Zillacode's problems, as on Day 12.
* 🔬 **See how Spark runs it, and make it cheaper**: the plan Spark builds for your code, what each part of it costs, and how to change the code so that Spark reads, shuffles and sorts less. The theory days explained Catalyst, AQE and shuffles; here you watch them work on your own queries.

Today:
1. 🔬 Reading a query plan: `explain()`, stages, tasks and the Spark UI
2. 🔬 What Catalyst does for you (pushdown, pruning), and what stops it
3. Dates and timestamps
4. Aggregations: `groupBy(...).agg(...)`
5. 🔬 How Spark runs a `groupBy`: partial aggregation, shuffles, distinct counts
6. Aggregate first, then join: the fan-out trap
7. Pivot
8. Arrays: `split`, `explode` and `collect_list`

In Zillacode's topic list this is *Datetime Operations*, *Aggregate Functions* and *Pivot Function*. On the optimization side it covers theory Days 2 and 3: the four plans, physical operators, predicate pushdown, column pruning, partial aggregation and AQE's partition coalescing.

## How to work
* Run the three **Setup** cells first, then the cell that prints the tables.
* **Lessons** explain one tool at a time. Run every example cell (its output is already shown under it), then change it and run it again until you can predict the result.
* Exercises come right after each lesson, in three kinds:
  * **`etl(...)`**: Zillacode's format, as on Day 12. Complete `etl` so that it **returns** a DataFrame that matches the expected output.
  * **🔍 Read the plan**: the code is given. Run it, read the plan (and the Spark UI when asked), and write your answer as a comment in the cell. When the question says *predict*, answer before you run.
  * **⚡ Make it faster**: write an `etl` that returns the expected output **and** whose plan meets the target under it (fewer shuffles, filters pushed into the file scan...). Your plan's column ids (`#58`) and paths differ; the operators must match.
* Keep the **Spark UI** open in a browser tab (the setup prints its address, normally http://localhost:4040). Its *SQL / DataFrame* tab shows every query's plan with row counts; the *Stages* tab shows tasks and shuffles.
* The answers are collapsed at the very end of the notebook. Open one only after you have tried.
* Today: about 1 hour 45 minutes of lessons and 2 hours of exercises (12 `etl`, 5 🔍 and 3 ⚡).

**Setup (once):** Java 17 or 21 and Python 3.10 or newer. In a new folder: `python3.12 -m venv .venv`, `source .venv/bin/activate`, `pip install pyspark==4.2.0 jupyterlab`, then `jupyter lab` and open this notebook. The notebook writes small Parquet files into a `data/` folder next to it.

## The data
A small online shop, the same as on Day 12. The setup cell creates every table from rows you can read, saves it as Parquet in `data/shop/` and reads it back, so the plans show real file scans. The cell after the setup prints the tables.

* **customers**: *customer_id*, *name*, *email*, *city*, *signup_date*
* **products**: *product_id*, *product_name*, *category*, *price* (dollars for one item)
* **orders**: *order_id*, *customer_id*, *product_id*, *quantity*, *order_date*, *status* (`delivered`, `shipped` or `cancelled`)
* **reviews**: *review_id*, *product_id*, *customer_id*, *review_date* (text like `15/01/2024`), *rating* (1 to 5), *comment* (messy text), *tags* (text like `fast delivery;good price`)
* **signups_raw**: sign-ups exactly as a web form sent them: *signup_id*, *full_name*, *email*, *phone*, *signup_date* (text in three formats), *amount* (text), *prefs* (JSON text)
* **customers_new**: the customers table one week later
* **promotions**: *promo_id*, *category*, *start_date*, *end_date* (both days included), *discount_pct*
* **events**: website visits: *customer_id*, *event_time* (a timestamp), *page*
* **referrals**: *referrer_id* (the customer who invited someone), *referred_id* (the customer who joined), *referral_date*

The data has the traps that Zillacode's hidden tests like to use: order 1011 is a guest checkout (no *customer_id*); customer 6 has no email and no city; customer 7 has never ordered; product 107 has never been sold; order 1016 is for product 108, which is not in *products*; review 6 has no tags; two events were loaded twice; referral 5 points to customer 9, who doesn't exist.

## Setup
Run these cells once. The first starts Spark and creates the tables. The second defines `measure()`, which runs a query and prints what it cost (Lesson 1 explains it).

In [ ]:
# Setup: starts Spark, creates the tables and saves them as Parquet files. Run it once.
import logging, os, sys
from datetime import date
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this notebook's Python
from pyspark.sql import SparkSession, functions as F, Window as W   # the same names as in Zillacode's editor

spark = (SparkSession.builder
         .master("local[4]")                               # 4 cores: up to 4 tasks run at the same time
         .appName("spark-lab-13")
         .config("spark.driver.memory", "2g")              # the default 1 GB is tight for 10 million rows
         .config("spark.sql.ansi.enabled", "false")        # behave like Zillacode (Spark 3): bad text gives null, not an error
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.driver.host", "localhost")         # run on this machine only; the Spark UI is at http://localhost:4040
         .config("spark.sql.maxMetadataStringLength", "1000")   # explain() prints long filter and column lists in full
         .config("spark.hadoop.parquet.hadoop.vectored.io.enabled", "false")   # Spark 4.2: without this, the UI's "Input" misses most bytes read
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.log.level", "FATAL")               # hide Spark's log messages
         # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this
         .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
         .getOrCreate())
for name in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
    logging.getLogger(name).setLevel(logging.CRITICAL)        # errors still show, without an extra JSON log

customers = spark.createDataFrame([
    # customer_id, name, email, city, signup_date
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Berlin",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     None,               None,       date(2024, 3, 10)),
    (7, "Grace Lee",        "grace@web.de",     "Berlin",   date(2024, 3, 15)),
], "customer_id INT, name STRING, email STRING, city STRING, signup_date DATE")

products = spark.createDataFrame([
    # product_id, product_name, category, price (dollars for one item)
    (101, "Laptop",       "Electronics", 1200.0),
    (102, "Headphones",   "Electronics",  150.0),
    (103, "Coffee Maker", "Kitchen",       85.0),
    (104, "Blender",      "Kitchen",       60.0),
    (105, "Novel",        "Books",         15.0),
    (106, "Cookbook",     "Books",         25.0),
    (107, "Desk Lamp",    "Home",          40.0),
], "product_id INT, product_name STRING, category STRING, price DOUBLE")

orders = spark.createDataFrame([
    # order_id, customer_id, product_id, quantity, order_date, status
    (1001, 1,    101, 1, date(2024, 1, 10), "delivered"),
    (1002, 1,    105, 3, date(2024, 1, 10), "delivered"),
    (1003, 2,    102, 2, date(2024, 1, 25), "delivered"),
    (1004, 3,    103, 1, date(2024, 2, 3),  "cancelled"),
    (1005, 2,    106, 1, date(2024, 2, 5),  "delivered"),
    (1006, 4,    104, 2, date(2024, 2, 15), "shipped"),
    (1007, 1,    102, 1, date(2024, 2, 20), "delivered"),
    (1008, 5,    101, 1, date(2024, 3, 2),  "shipped"),
    (1009, 3,    105, 4, date(2024, 3, 5),  "delivered"),
    (1010, 4,    106, 2, date(2024, 3, 8),  "cancelled"),
    (1011, None, 104, 1, date(2024, 3, 9),  "delivered"),
    (1012, 6,    103, 2, date(2024, 3, 12), "shipped"),
    (1013, 2,    101, 1, date(2024, 3, 18), "delivered"),
    (1014, 5,    105, 5, date(2024, 3, 20), "delivered"),
    (1015, 1,    103, 1, date(2024, 3, 25), "shipped"),
    (1016, 3,    108, 1, date(2024, 3, 28), "delivered"),
], "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING")

reviews = spark.createDataFrame([
    # review_id, product_id, customer_id, review_date (text), rating, comment, tags (text)
    (1, 101, 1, "15/01/2024", 5, "  Great laptop, very FAST!  ",    "fast delivery;good price"),
    (2, 105, 1, "20/01/2024", 4, "Nice story.",                     "good price"),
    (3, 102, 2, "02/02/2024", 3, "Sound is ok, delivery was slow",  "slow delivery"),
    (4, 103, 3, "10/02/2024", 1, "Broke after one week!!",          "broken;slow delivery"),
    (5, 101, 5, "09/03/2024", 5, "Fast and powerful",               "fast delivery"),
    (6, 105, 3, "12/03/2024", 2, "Too long, boring",                None),
    (7, 106, 2, "25/03/2024", 4, "Useful recipes",                  "good price;gift"),
    (8, 103, 6, "30/03/2024", 4, "Good coffee, slow start",         "gift"),
], "review_id INT, product_id INT, customer_id INT, review_date STRING, rating INT, comment STRING, tags STRING")

signups_raw = spark.createDataFrame([
    # signup_id, full_name, email, phone, signup_date (text, three formats), amount (text), prefs (JSON text)
    (1, "  aziz   KARIMOV ", "Aziz@Mail.UZ ",      "+998 90 123-45-67", "2024-01-05", "12.50", '{"newsletter": true, "lang": "uz"}'),
    (2, "bella schmidt",     "bella@web.de",       "0049 151 2345678",  "20/01/2024", "abc",   '{"newsletter": false, "lang": "de"}'),
    (3, "Carlos  Ruiz",      "",                   "(34) 612 345 678",  "2024/02/02", "7",     '{"lang": "es"}'),
    (4, "DILNOZA rahimova",  "dilnoza@mail.uz",    "90-765-43-21",      "14/02/2024", "",      None),
    (5, "Emma Wilson",       "emma[at]post.co.uk", "07700 900123",      "2024-03-01", "3.75",  '{"newsletter": true}'),
    (6, "aziz karimov",      "aziz@mail.uz",       "+998901234567",     "2024-03-02", "1.00",  '{"newsletter": false, "lang": "uz"}'),
], "signup_id INT, full_name STRING, email STRING, phone STRING, signup_date STRING, amount STRING, prefs STRING")

customers_new = spark.createDataFrame([
    # the customers table one week later: 2 moved, 6 has an email now, 7 is gone, 8 is new
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Munich",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     "farid@mail.az",    None,       date(2024, 3, 10)),
    (8, "Hana Sato",        "hana@mail.jp",     "Osaka",    date(2024, 4, 2)),
], customers.schema)

promotions = spark.createDataFrame([
    # promo_id, category, start_date, end_date (both days included), discount_pct
    ("WINTER",   "Electronics", date(2024, 1, 1),  date(2024, 1, 31), 10),
    ("BOOKWEEK", "Books",       date(2024, 3, 1),  date(2024, 3, 10), 20),
    ("SPRING",   "Kitchen",     date(2024, 3, 1),  date(2024, 3, 31), 15),
    ("FLASH",    "Electronics", date(2024, 3, 15), date(2024, 3, 20), 5),
], "promo_id STRING, category STRING, start_date DATE, end_date DATE, discount_pct INT")

events = spark.createDataFrame([
    # customer_id, event_time, page (two rows were loaded twice)
    (1, "2024-03-01 09:00:00", "home"),
    (1, "2024-03-01 09:10:00", "product"),
    (1, "2024-03-01 09:10:00", "product"),
    (1, "2024-03-01 11:00:00", "home"),
    (1, "2024-03-02 10:00:00", "cart"),
    (1, "2024-03-03 10:30:00", "home"),
    (1, "2024-03-05 08:00:00", "home"),
    (2, "2024-03-01 20:00:00", "home"),
    (2, "2024-03-01 20:20:00", "product"),
    (2, "2024-03-01 20:45:00", "cart"),
    (2, "2024-03-04 18:00:00", "home"),
    (2, "2024-03-05 18:00:00", "home"),
    (2, "2024-03-05 18:00:00", "home"),
    (3, "2024-03-02 12:00:00", "product"),
    (3, "2024-03-03 12:00:00", "product"),
    (3, "2024-03-04 12:00:00", "cart"),
], "customer_id INT, event_time STRING, page STRING").withColumn("event_time", F.to_timestamp("event_time"))

referrals = spark.createDataFrame([
    # referrer_id (the customer who invited), referred_id (the customer who joined), referral_date
    (1, 4, date(2024, 2, 10)),
    (4, 5, date(2024, 2, 25)),
    (1, 6, date(2024, 3, 8)),
    (2, 7, date(2024, 3, 12)),
    (2, 9, date(2024, 3, 30)),
], "referrer_id INT, referred_id INT, referral_date DATE")

# Save every table as Parquet and read it back. In-memory tables would show "Scan ExistingRDD" in plans;
# Parquet files show what real jobs read: "FileScan parquet" with the columns and filters it reads.
for name, df in [("customers", customers), ("products", products), ("orders", orders), ("reviews", reviews),
                 ("signups_raw", signups_raw), ("customers_new", customers_new), ("promotions", promotions),
                 ("events", events), ("referrals", referrals)]:
    df.coalesce(1).write.mode("overwrite").parquet(f"data/shop/{name}")   # one file per table
customers = spark.read.parquet("data/shop/customers")
products = spark.read.parquet("data/shop/products")
orders = spark.read.parquet("data/shop/orders")
reviews = spark.read.parquet("data/shop/reviews")
signups_raw = spark.read.parquet("data/shop/signups_raw")
customers_new = spark.read.parquet("data/shop/customers_new")
promotions = spark.read.parquet("data/shop/promotions")
events = spark.read.parquet("data/shop/events")
referrals = spark.read.parquet("data/shop/referrals")

print("Spark", spark.version, "is ready. Spark UI:", spark.sparkContext.uiWebUrl)
print("Tables: customers, products, orders, reviews, signups_raw, customers_new, promotions, events, referrals")

Spark 4.2.0 is ready. Spark UI: http://localhost:4040
Tables: customers, products, orders, reviews, signups_raw, customers_new, promotions, events, referrals


In [ ]:
# measure(): run a query and print what it cost. These are the numbers the Spark UI shows.
import json, time, urllib.request

def ui(path):
    """Read one page of the Spark UI's REST API (the data behind http://localhost:4040)."""
    sc = spark.sparkContext
    with urllib.request.urlopen(f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/{path}") as response:
        return json.load(response)

def size(n):
    for unit, k in (("GB", 1e9), ("MB", 1e6), ("KB", 1e3)):
        if n >= k:
            return f"{n / k:.1f} {unit}"
    return f"{n} B"

def measure(query, label="", stages=False, plan=False):
    """Run `query` to the end, then print: time, jobs, stages, tasks, bytes read from files, shuffled and spilled.

    query:  a DataFrame (it runs with the "noop" writer: every task runs, nothing is kept or shown),
            or a function without arguments, like  lambda: df.count()
    stages: also print one line per stage: its tasks, the median and the slowest task, its shuffle
    plan:   also print the final plan of the query, after AQE's changes (the plan in the UI's SQL tab)
    """
    sc = spark.sparkContext
    group = f"measure-{time.time_ns()}"
    sc.setJobGroup(group, label or "measure")
    start = time.perf_counter()
    try:
        if callable(query):
            query()
        else:
            query.write.format("noop").mode("overwrite").save()
    finally:
        sc.setLocalProperty("spark.jobGroup.id", None)
    seconds = time.perf_counter() - start
    tracker = sc.statusTracker()
    jobs = sorted(tracker.getJobIdsForGroup(group))
    stage_ids = sorted({s for j in jobs for s in tracker.getJobInfo(j).stageIds})
    for _ in range(100):                     # the UI records a stage a moment after it ends
        info = [ui(f"stages/{s}")[0] for s in stage_ids]
        if all(s["status"] in ("COMPLETE", "SKIPPED", "FAILED") for s in info):
            break
        time.sleep(0.05)
    ran = [s for s in info if s["status"] == "COMPLETE"]       # "SKIPPED": output reused, did not run
    total = lambda key: sum(s[key] for s in ran)
    print(f"{label + ': ' if label else ''}{seconds:.2f} s | {len(jobs)} jobs, {len(ran)} stages, "
          f"{total('numCompleteTasks')} tasks | read {size(total('inputBytes'))} | "
          f"shuffle {size(total('shuffleWriteBytes'))} | spill {size(total('diskBytesSpilled'))}")
    if stages:
        for s in ran:
            t = ui(f"stages/{s['stageId']}/{s['attemptId']}/taskSummary?quantiles=0.5,1.0")["executorRunTime"]
            print(f"   stage {s['stageId']:>3}: {s['numCompleteTasks']:>3} tasks, median {t[0] / 1000:.2f} s, "
                  f"max {t[1] / 1000:.2f} s | read {size(s['inputBytes'])} | "
                  f"shuffle read {size(s['shuffleReadBytes'])}, write {size(s['shuffleWriteBytes'])}")
    if plan:
        runs = [e for e in ui("sql?offset=0&length=1000000") if set(e["successJobIds"]) & set(jobs)]
        tree = ui(f"sql/{runs[-1]['id']}?planDescription=true")["planDescription"].split("\n\n")[0]
        print(tree.replace(", Statistics(sizeInBytes=8.0 EiB)", ""))

print("measure() is ready")

measure() is ready


Print every table, so you can check results by eye:

In [ ]:
for name in ["customers", "products", "orders", "reviews", "signups_raw", "customers_new", "promotions", "events", "referrals"]:
    print(name)
    globals()[name].show(truncate=False)

customers
+-----------+----------------+----------------+--------+-----------+
|customer_id|name            |email           |city    |signup_date|
+-----------+----------------+----------------+--------+-----------+
|1          |Aziz Karimov    |aziz@mail.uz    |Tashkent|2024-01-05 |
|2          |Bella Schmidt   |bella@web.de    |Berlin  |2024-01-20 |
|3          |Carlos Ruiz     |carlos@correo.es|Madrid  |2024-02-02 |
|4          |Dilnoza Rahimova|dilnoza@mail.uz |Tashkent|2024-02-14 |
|5          |Emma Wilson     |emma@post.co.uk |London  |2024-03-01 |
|6          |Farid Aliyev    |NULL            |NULL    |2024-03-10 |
|7          |Grace Lee       |grace@web.de    |Berlin  |2024-03-15 |
+-----------+----------------+----------------+--------+-----------+

products
+----------+------------+-----------+------+
|product_id|product_name|category   |price |
+----------+------------+-----------+------+
|101       |Laptop      |Electronics|1200.0|
|102       |Headphones  |Electronics|150.

## Lesson 1 · 🔬 Reading a query plan

Your code says **what** you want. When an action runs (`show`, `collect`, `count`, a write), Spark decides **how**: Catalyst rewrites the query (theory Day 2), picks an algorithm for each step, and AQE adjusts the plan while it runs (Day 3). The result is the **physical plan**. It tells you exactly what will happen: which files and columns are read, where rows are shuffled between tasks, and which algorithms run. Every optimization starts by reading it.

`df.explain(mode)` prints the plan **without running anything** ([docs](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.explain.html); in SQL, [`EXPLAIN`](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-explain.html)):

| `mode` | Prints |
|---|---|
| `"simple"` (the default: `explain()`) | the physical plan |
| `"extended"` (also `explain(True)`) | four plans: parsed, analyzed, optimized and physical |
| `"formatted"` | a short tree of numbered operators, then the details of each one |
| `"cost"` | the optimized plan with Spark's size estimates (Day 14) |
| `"codegen"` | the Java code Spark generates (rarely needed) |

**The four plans.** *Parsed*: your code as written. *Analyzed*: names and types resolved against the tables (a typo fails here). *Optimized*: after Catalyst's rewrite rules (Lesson 2). *Physical*: the operators that will run, with an algorithm chosen for each step. You'll read the physical plan most.

**How to read a physical plan**
* It's a tree printed top-down, but the data flows **bottom-up**: from the file scan at the bottom to the result at the top. `+-` marks a child. A join has two children; `:-` marks the first (left) one.
* `*(1)` in front of operators means **whole-stage code generation** (Tungsten, Day 2) fused them into one Java function; the same number means the same function. An operator without a star runs on its own.
* In `quantity#58`, `#58` is the column's internal id. These ids and the `plan_id` numbers differ on your machine.
* `AdaptiveSparkPlan isFinalPlan=false` at the top means AQE is on and what's below is the **initial** plan. Once the query has run, `explain()` on the same DataFrame shows the final plan (`isFinalPlan=true`).

**The operators of today's queries** (joins come on Day 14):

| Operator | What it does | What to look at |
|---|---|---|
| `FileScan parquet [columns]` | reads the files | `ReadSchema`: the columns it reads; `PushedFilters`: the conditions the Parquet reader checks itself (Lesson 2); `Location`: the folder |
| `ColumnarToRow` | turns Parquet's column batches into rows | nothing: it always follows a Parquet scan |
| `Filter`, `Project` | keep the rows that pass a condition / compute and choose columns | |
| `HashAggregate` | a `groupBy` step, usually twice: with `partial_` functions before the shuffle, final after it (Lesson 5) | |
| **`Exchange`** | a **shuffle**: every row is sent to the partition that owns it | the expensive part. `hashpartitioning(key, 200)`: by key, into 200 partitions; `rangepartitioning`: by sorted ranges, for `orderBy`; `SinglePartition`: everything into one partition |
| `Sort` | sorts the rows inside each partition | after an `Exchange rangepartitioning` it's a global sort |
| `TakeOrderedAndProject` | `orderBy(...).limit(n)`: keeps the top n of each partition, then the top n of those | no shuffle of the whole table |
| `CollectLimit` | `limit(n)`, and `show()`: stops after n rows | `show()` doesn't run the whole query |
| `AQEShuffleRead coalesced` | AQE merged small shuffle partitions after it saw their real size | only in the final plan |

**From the plan to stages and tasks.** Spark cuts the physical plan at every `Exchange`. The operators between two exchanges form one **stage**, and a stage runs as **one task per partition** (here up to 4 tasks at a time: `local[4]`). So the exchanges in a plan tell you how many shuffles the query needs before you run it, and the number of shuffles is usually the first thing to reduce.

**Watching it run: the Spark UI** ([guide](https://spark.apache.org/docs/latest/web-ui.html)), at the address the setup printed:
* **SQL / DataFrame** tab: one line per query. Click one to see its plan as a diagram with the real numbers of every operator (rows out, sizes, time), and the final plan as text at the bottom.
* **Jobs** and **Stages** tabs: every stage's tasks, input, shuffle read and write, and spill, with the min, median and max per task.

**`measure(query)`** (second setup cell) runs a query to the end and prints the same numbers on one line: seconds, jobs, stages, tasks, bytes read from files, bytes shuffled and bytes spilled to disk. `stages=True` adds a line per stage; `plan=True` adds the final plan. A DataFrame runs with Spark's `noop` writer: every task runs, but nothing is collected or printed. Two things to know when you read it: with AQE, Spark submits **each shuffle stage as its own job**, so a query with one shuffle shows 2 jobs; and on tiny data the seconds mean nothing, while the stages, tasks and bytes do.

In [ ]:
# The physical plan of a filter and a select. Read it from the bottom: scan, filter, project
q = orders.filter(F.col("quantity") >= 2).select("order_id", "status")
q.explain()

== Physical Plan ==
*(1) Project [order_id#55, status#60]
+- *(1) Filter (isnotnull(quantity#58) AND (quantity#58 >= 2))
   +- *(1) ColumnarToRow
      +- FileScan parquet [order_id#55,quantity#58,status#60] Batched: true, DataFilters: [isnotnull(quantity#58), (quantity#58 >= 2)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/shop/orders], PartitionFilters: [], PushedFilters: [IsNotNull(quantity), GreaterThanOrEqual(quantity,2)], ReadSchema: struct<order_id:int,quantity:int,status:string>




In [ ]:
# The same plan in "formatted" mode: a short tree of numbered operators, then the details of each
q.explain(mode="formatted")

== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [order_id#55, quantity#58, status#60]
Batched: true
Location: InMemoryFileIndex [file:/Users/you/spark-lab/data/shop/orders]
PushedFilters: [IsNotNull(quantity), GreaterThanOrEqual(quantity,2)]
ReadSchema: struct<order_id:int,quantity:int,status:string>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [order_id#55, quantity#58, status#60]

(3) Filter [codegen id : 1]
Input [3]: [order_id#55, quantity#58, status#60]
Condition : (isnotnull(quantity#58) AND (quantity#58 >= 2))

(4) Project [codegen id : 1]
Output [2]: [order_id#55, status#60]
Input [3]: [order_id#55, quantity#58, status#60]




In [ ]:
# A groupBy: partial aggregate, shuffle (Exchange), final aggregate. AQE wraps the plan
per_customer = orders.groupBy("customer_id").agg(F.count("*").alias("orders"), F.sum("quantity").alias("items"))
per_customer.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[customer_id#56], functions=[count(1), sum(quantity#58)])
   +- Exchange hashpartitioning(customer_id#56, 200), ENSURE_REQUIREMENTS, [plan_id=354]
      +- HashAggregate(keys=[customer_id#56], functions=[partial_count(1), partial_sum(quantity#58)])
         +- FileScan parquet [customer_id#56,quantity#58] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/shop/orders], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<customer_id:int,quantity:int>




In [ ]:
# After an action, the same DataFrame shows AQE's final plan: the 200 shuffle partitions were coalesced
per_customer.collect()
per_customer.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- *(2) HashAggregate(keys=[customer_id#56], functions=[count(1), sum(quantity#58)])
      +- AQEShuffleRead coalesced
         +- ShuffleQueryStage 0
            +- Exchange hashpartitioning(customer_id#56, 200), ENSURE_REQUIREMENTS, [plan_id=368]
               +- *(1) HashAggregate(keys=[customer_id#56], functions=[partial_count(1), partial_sum(quantity#58)])
                  +- *(1) ColumnarToRow
                     +- FileScan parquet [customer_id#56,quantity#58] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/shop/orders], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<customer_id:int,quantity:int>
+- == Initial Plan ==
   HashAggregate(keys=[customer_id#56], functions=[count(1), sum(quantity#58)])
   +- Exchange hashpartitioning(customer_id#56, 200), ENSURE_REQUIREMENTS, [plan_id=354]
      +- Hash

In [ ]:
# Measured: one task reads the file, one task aggregates after the shuffle
measure(per_customer, "groupBy", stages=True)
# Without AQE, the shuffle keeps all 200 partitions: 200 tasks for 7 groups
spark.conf.set("spark.sql.adaptive.enabled", "false")
measure(per_customer, "groupBy, AQE off", stages=True)
spark.conf.set("spark.sql.adaptive.enabled", "true")

groupBy: 0.08 s | 2 jobs, 2 stages, 2 tasks | read 3.7 KB | shuffle 461 B | spill 0 B
   stage  30:   1 tasks, median 0.01 s, max 0.01 s | read 3.7 KB | shuffle read 0 B, write 461 B
   stage  32:   1 tasks, median 0.00 s, max 0.00 s | read 0 B | shuffle read 461 B, write 0 B
groupBy, AQE off: 0.28 s | 1 jobs, 2 stages, 201 tasks | read 3.7 KB | shuffle 461 B | spill 0 B
   stage  33:   1 tasks, median 0.02 s, max 0.02 s | read 3.7 KB | shuffle read 0 B, write 461 B
   stage  34: 200 tasks, median 0.00 s, max 0.01 s | read 0 B | shuffle read 461 B, write 0 B


### Exercises

**Q1 · 🔍 Read the plan** **Predict**, then check: how many `Exchange` operators does this query have, and what does each one do? How many stages will it run? Read the plan first, then run `measure` and look at the *Stages* tab.

In [ ]:
q1 = orders.groupBy("status").agg(F.sum("quantity").alias("items")).orderBy(F.desc("items"))
q1.explain()
measure(q1, "q1", stages=True)
# Your answer:

**Q2 · 🔍 Read the plan** Which columns does Spark read from each table for this query? Where in the plan is the status condition checked?

In [ ]:
q2 = (customers.join(orders, "customer_id")
               .filter(F.col("status") == "cancelled")
               .select("name", "order_date"))
q2.explain()
# Your answer:

**Q3 · 🔍 Read the plan** **Predict**: how many shuffles does this query need, and why? Then run it and check with the stages that `measure` prints (or the SQL / DataFrame tab of the Spark UI).

In [ ]:
measure(lambda: orders.select("status").distinct().count(), "distinct statuses", stages=True)
# Your answer:

## Lesson 2 · 🔬 What Catalyst does for you, and what stops it

Before anything runs, Catalyst rewrites the query with rules that are always safe ([the rules in Spark's source](https://github.com/apache/spark/blob/master/sql/catalyst/src/main/scala/org/apache/spark/sql/catalyst/optimizer/Optimizer.scala)). You can watch it: compare the *analyzed* plan (what you wrote) with the *optimized* one. The rules you'll see most:

| Rule | What it does | Where you see it |
|---|---|---|
| **Predicate pushdown** | moves every filter as early as it can: below `withColumn` and `select`, below a join (into the side the column comes from), and into the file scan | a `Filter` right above the scan; `PushedFilters` in `FileScan` |
| **Column pruning** | reads only the columns the query uses. Parquet stores each column separately, so the others are never read from disk | `ReadSchema` |
| **Constant folding** | computes constant parts once: `F.lit(1) + F.lit(1)` becomes `2` | the optimized plan |
| **Combining filters and projections** | merges consecutive `filter`s into one condition, consecutive `select`s into one | one `Filter`, one `Project` |
| **Inferring not-null filters** | adds `isnotnull(col)` for a column used in a comparison or a join key: null can't pass `=`, `>`... anyway | `isnotnull(...)` in conditions |

So the order of your `filter` and `withColumn` calls doesn't change what runs: write the order that reads best.

**Three lists in `FileScan`:**
* `DataFilters`: the conditions on the table's columns. Spark checks them in the `Filter` operator after reading.
* `PushedFilters`: the part of them the **Parquet reader** understands. Parquet stores the min and max of every column for each block of rows (a *row group*), and the reader skips the blocks whose range can't match (theory Day 4). Spark's `Filter` still checks every row afterwards, so pushing a filter never changes the result; it only reads less.
* `PartitionFilters`: conditions on folder partition columns (`.../month=2024-03/`), which skip whole folders (Day 15).

**What stops pushdown into the file.** The Parquet reader understands a **plain column** compared with a **constant**: `=`, `<`, `>=`, `between`, `isin`, `isNull`, `startswith`, `endswith`, `contains`. A **function around the column** hides it:

| Written as | Pushed into the scan? | A pushable way to write it |
|---|---|---|
| `F.year("order_date") == 2024` | no | `F.col("order_date").between("2024-01-01", "2024-12-31")` |
| `F.month("order_date") == 3`, `F.date_format("order_date", "yyyy-MM") == "2024-03"` | no | `(F.col("order_date") >= "2024-03-01") & (F.col("order_date") < "2024-04-01")` |
| `F.upper("city") == "BERLIN"` | no | `F.col("city") == "Berlin"`, when the data is stored that way |
| a Python UDF in the condition | no | built-in functions (Day 15) |

Spark still applies these conditions correctly; it just has to read every row group first. On Zillacode's tiny tables none of this changes the result or the time. On a table of a few terabytes, it decides between reading all of it and reading one month of it.

In [ ]:
# Written in an awkward order: a computed column first, two filters, a select in between
messy = (orders.withColumn("double_qty", F.col("quantity") * (F.lit(1) + F.lit(1)))
               .filter(F.col("status") == "delivered")
               .select("order_id", "double_qty", "quantity")
               .filter(F.col("quantity") > 1))
messy.explain(mode="extended")

== Parsed Logical Plan ==
'Filter '`>`('quantity, 1)
+- Project [order_id#55, double_qty#282, quantity#58]
   +- Filter (status#60 = delivered)
      +- Project [order_id#55, customer_id#56, product_id#57, quantity#58, order_date#59, status#60, (quantity#58 * (1 + 1)) AS double_qty#282]
         +- Relation [order_id#55,customer_id#56,product_id#57,quantity#58,order_date#59,status#60] parquet

== Analyzed Logical Plan ==
order_id: int, double_qty: int, quantity: int
Filter (quantity#58 > 1)
+- Project [order_id#55, double_qty#282, quantity#58]
   +- Filter (status#60 = delivered)
      +- Project [order_id#55, customer_id#56, product_id#57, quantity#58, order_date#59, status#60, (quantity#58 * (1 + 1)) AS double_qty#282]
         +- Relation [order_id#55,customer_id#56,product_id#57,quantity#58,order_date#59,status#60] parquet

== Optimized Logical Plan ==
Project [order_id#55, (quantity#58 * 2) AS double_qty#282, quantity#58]
+- Filter ((isnotnull(status#60) AND isnotnull(quantity#58)

In [ ]:
# A filter written after a join runs before it, on the side its column comes from
(orders.join(products, "product_id")
       .filter((F.col("category") == "Books") & (F.col("quantity") > 1))
       .select("order_id", "product_name")
       .explain())

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [order_id#55, product_name#52]
   +- BroadcastHashJoin [product_id#57], [product_id#51], Inner, BuildLeft, false, false
      :- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[1, int, true] as bigint)),false), [plan_id=739]
      :  +- Project [order_id#55, product_id#57]
      :     +- Filter ((isnotnull(quantity#58) AND (quantity#58 > 1)) AND isnotnull(product_id#57))
      :        +- FileScan parquet [order_id#55,product_id#57,quantity#58] Batched: true, DataFilters: [isnotnull(quantity#58), (quantity#58 > 1), isnotnull(product_id#57)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/shop/orders], PartitionFilters: [], PushedFilters: [IsNotNull(quantity), GreaterThan(quantity,1), IsNotNull(product_id)], ReadSchema: struct<order_id:int,product_id:int,quantity:int>
      +- Project [product_id#51, product_name#52]
         +- Filter ((isnotnull(category#53) AND (cate

In [ ]:
# The same rows, two plans: a function around the column hides it from the Parquet reader
orders.filter(F.month("order_date") == 3).select("order_id").explain()
orders.filter((F.col("order_date") >= "2024-03-01") & (F.col("order_date") < "2024-04-01")).select("order_id").explain()

== Physical Plan ==
*(1) Project [order_id#55]
+- *(1) Filter (isnotnull(order_date#59) AND (month(order_date#59) = 3))
   +- *(1) ColumnarToRow
      +- FileScan parquet [order_id#55,order_date#59] Batched: true, DataFilters: [isnotnull(order_date#59), (month(order_date#59) = 3)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/shop/orders], PartitionFilters: [], PushedFilters: [IsNotNull(order_date)], ReadSchema: struct<order_id:int,order_date:date>


== Physical Plan ==
*(1) Project [order_id#55]
+- *(1) Filter ((isnotnull(order_date#59) AND (order_date#59 >= 2024-03-01)) AND (order_date#59 < 2024-04-01))
   +- *(1) ColumnarToRow
      +- FileScan parquet [order_id#55,order_date#59] Batched: true, DataFilters: [isnotnull(order_date#59), (order_date#59 >= 2024-03-01), (order_date#59 < 2024-04-01)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/shop/orders], PartitionFilters: [], PushedFilters: [IsNotNull(orde

In [ ]:
# Text conditions: upper() hides the column, endswith() is pushed as StringEndsWith
customers.filter(F.upper("city") == "BERLIN").select("name").explain()
customers.filter(F.col("email").endswith(".de")).select("name").explain()

== Physical Plan ==
*(1) Project [name#47]
+- *(1) Filter (isnotnull(city#49) AND (upper(city#49) = BERLIN))
   +- *(1) ColumnarToRow
      +- FileScan parquet [name#47,city#49] Batched: true, DataFilters: [isnotnull(city#49), (upper(city#49) = BERLIN)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/shop/customers], PartitionFilters: [], PushedFilters: [IsNotNull(city)], ReadSchema: struct<name:string,city:string>


== Physical Plan ==
*(1) Project [name#47]
+- *(1) Filter (isnotnull(email#48) AND EndsWith(email#48, .de))
   +- *(1) ColumnarToRow
      +- FileScan parquet [name#47,email#48] Batched: true, DataFilters: [isnotnull(email#48), EndsWith(email#48, .de)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/shop/customers], PartitionFilters: [], PushedFilters: [IsNotNull(email), StringEndsWith(email,.de)], ReadSchema: struct<name:string,email:string>




### Exercises

**Q4 · 🔍 Read the plan** Which of the three conditions reach the Parquet reader (`PushedFilters`), and which are only checked by Spark's `Filter` after reading? How would you write the one that isn't pushed so that it is?

In [ ]:
q4 = customers.filter((F.col("signup_date") >= "2024-02-01")
                      & F.lower("email").endswith(".uz")
                      & F.col("city").isNotNull())
q4.explain()
# Your answer:

**Q5 · ⚡ `etl(orders)`** Show the delivered orders placed in March 2024: order_id, customer_id and order_date, sorted by order_id. Write the conditions so that the Parquet reader gets all of them.

Expected output:

```
+--------+-----------+----------+
|order_id|customer_id|order_date|
+--------+-----------+----------+
|    1009|          3|2024-03-05|
|    1011|       NULL|2024-03-09|
|    1013|          2|2024-03-18|
|    1014|          5|2024-03-20|
|    1016|          3|2024-03-28|
+--------+-----------+----------+
```

**Target:** `PushedFilters` contains `EqualTo(status,delivered)`, `GreaterThanOrEqual(order_date,2024-03-01)` and a `LessThan` (or `LessThanOrEqual`) on order_date.

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()
etl(orders).explain()

## Lesson 3 · Dates and timestamps

Spark has a `date` type (`2024-03-02`), a `timestamp` type (`2024-03-02 09:10:00`), and then there's text that only *looks* like a date. **In Zillacode every date arrives as text** (Day 12, Lesson 0), so the first step is usually a conversion:

| Function | What it does |
|---|---|
| `F.to_date(col)` | text in the form `yyyy-MM-dd` to a date |
| `F.to_date(col, "dd/MM/yyyy")` | text in another form to a date: the pattern describes the **text** |
| `F.to_timestamp(col, "yyyy-MM-dd HH:mm:ss")` | text to a timestamp |
| `F.date_format(col, "yyyy-MM")` | a date or timestamp to text, in any pattern |

**Pattern letters** ([full list](https://spark.apache.org/docs/latest/sql-ref-datetime-pattern.html)). Case matters:

| Letters | Meaning | Example |
|---|---|---|
| `yyyy` | year | 2024 |
| `MM`, `M` | month with / without a leading zero | 03, 3 |
| `MMM`, `MMMM` | month name | Mar, March |
| `dd`, `d` | day of the month | 05, 5 |
| `HH` | hour, 0 to 23 | 18 |
| `mm` | **minutes** (not months!) | 45 |
| `ss` | seconds | 00 |
| `E`, `EEEE` | weekday name | Sat, Saturday |

`MM` vs `mm` is the classic bug: `"yyyy-mm-dd"` reads the month digits as minutes. Text that doesn't fit the pattern becomes **null** (on Zillacode, and here with ANSI off). An end date of `"ongoing"`, for example, turns into null, and you can then treat that null as "still running".

**Parts and date math**

| Function | Result |
|---|---|
| `F.year`, `F.month`, `F.dayofmonth`, `F.quarter`, `F.weekofyear` | parts as numbers |
| `F.dayofweek(d)` | **1 = Sunday**, 2 = Monday, ..., 7 = Saturday. Weekend: `F.dayofweek(d).isin(1, 7)` |
| `F.hour`, `F.minute`, `F.second` | parts of a timestamp |
| `F.datediff(end, start)` | whole days from start to end (end minus start) |
| `F.months_between(end, start)` | months, with a fraction |
| `F.date_add(d, n)`, `F.date_sub(d, n)`, `F.add_months(d, n)` | move a date by days or months |
| `F.last_day(d)`, `F.trunc(d, "month")` | the last / first day of the month (`trunc` also takes `"year"`) |
| `F.date_trunc("hour", ts)` | a timestamp cut down to the hour (or `"day"`, `"month"`, ...) |
| `F.unix_timestamp(ts)` | seconds since 1970. Subtract two of them for a duration in seconds; divide by 60 for minutes. |
| `F.current_date()` | today: avoid it on Zillacode, where problems state their own "today" ("assume the current year is 2024") |

A date column can be compared with a text literal: `F.col("order_date") >= "2024-03-01"` works, and `between("2024-03-01", "2024-03-31")` includes both ends.

On Zillacode: #1 (released in the last 5 years), #20 (a project's duration in days), #22 (minutes watched while a subscription was active, with `"ongoing"` end dates), #31 (weekend orders from `MM/DD/YYYY` text), and #8 and #35 (one row per date).

🔬 **In the plan.** To keep the rows of a period, compare the raw column with date text: `(F.col("order_date") >= "2024-03-01") & (F.col("order_date") < "2024-04-01")`. `F.month("order_date") == 3` keeps the same rows, but only the first form reaches the Parquet reader (Lesson 2). Spark converts the text to a date once, while planning: the plan shows `order_date >= 2024-03-01` with no conversion per row.

In [ ]:
# Text to date: the pattern must describe the text, or you get null
reviews.select(
    "review_date",
    F.to_date("review_date", "dd/MM/yyyy").alias("as_date"),
    F.to_date("review_date").alias("wrong_pattern"),
).show(4)

+-----------+----------+-------------+
|review_date|   as_date|wrong_pattern|
+-----------+----------+-------------+
| 15/01/2024|2024-01-15|         NULL|
| 20/01/2024|2024-01-20|         NULL|
| 02/02/2024|2024-02-02|         NULL|
| 10/02/2024|2024-02-10|         NULL|
+-----------+----------+-------------+
only showing top 4 rows


In [ ]:
day = F.to_date("review_date", "dd/MM/yyyy")
reviews.select(
    "review_id", day.alias("day"),
    F.year(day).alias("year"), F.month(day).alias("month"), F.dayofweek(day).alias("dayofweek"),
    F.date_format(day, "EEEE").alias("weekday"),
    F.dayofweek(day).isin(1, 7).alias("weekend"),
    F.date_format(day, "yyyy-MM").alias("year_month"),
).show()

+---------+----------+----+-----+---------+--------+-------+----------+
|review_id|       day|year|month|dayofweek| weekday|weekend|year_month|
+---------+----------+----+-----+---------+--------+-------+----------+
|        1|2024-01-15|2024|    1|        2|  Monday|  false|   2024-01|
|        2|2024-01-20|2024|    1|        7|Saturday|   true|   2024-01|
|        3|2024-02-02|2024|    2|        6|  Friday|  false|   2024-02|
|        4|2024-02-10|2024|    2|        7|Saturday|   true|   2024-02|
|        5|2024-03-09|2024|    3|        7|Saturday|   true|   2024-03|
|        6|2024-03-12|2024|    3|        3| Tuesday|  false|   2024-03|
|        7|2024-03-25|2024|    3|        2|  Monday|  false|   2024-03|
|        8|2024-03-30|2024|    3|        7|Saturday|   true|   2024-03|
+---------+----------+----+-----+---------+--------+-------+----------+



In [ ]:
orders.select(
    "order_id", "order_date",
    F.datediff(F.to_date(F.lit("2024-04-01")), "order_date").alias("days_until_april"),
    F.date_add("order_date", 30).alias("plus_30_days"),
    F.add_months("order_date", 1).alias("plus_1_month"),
    F.trunc("order_date", "month").alias("month_start"),
    F.last_day("order_date").alias("month_end"),
).show(5)

+--------+----------+----------------+------------+------------+-----------+----------+
|order_id|order_date|days_until_april|plus_30_days|plus_1_month|month_start| month_end|
+--------+----------+----------------+------------+------------+-----------+----------+
|    1001|2024-01-10|              82|  2024-02-09|  2024-02-10| 2024-01-01|2024-01-31|
|    1002|2024-01-10|              82|  2024-02-09|  2024-02-10| 2024-01-01|2024-01-31|
|    1003|2024-01-25|              67|  2024-02-24|  2024-02-25| 2024-01-01|2024-01-31|
|    1004|2024-02-03|              58|  2024-03-04|  2024-03-03| 2024-02-01|2024-02-29|
|    1005|2024-02-05|              56|  2024-03-06|  2024-03-05| 2024-02-01|2024-02-29|
+--------+----------+----------------+------------+------------+-----------+----------+
only showing top 5 rows


In [ ]:
# Durations between timestamps: unix_timestamp gives seconds
start = F.to_timestamp(F.lit("2024-03-01 20:00:00"))
events.filter(F.col("customer_id") == 2).select(
    "event_time",
    F.hour("event_time").alias("hour"),
    F.date_trunc("day", "event_time").alias("day_start"),
    ((F.unix_timestamp("event_time") - F.unix_timestamp(start)) / 60).alias("minutes_after_start"),
).show()

+-------------------+----+-------------------+-------------------+
|         event_time|hour|          day_start|minutes_after_start|
+-------------------+----+-------------------+-------------------+
|2024-03-01 20:00:00|  20|2024-03-01 00:00:00|                0.0|
|2024-03-01 20:20:00|  20|2024-03-01 00:00:00|               20.0|
|2024-03-01 20:45:00|  20|2024-03-01 00:00:00|               45.0|
|2024-03-04 18:00:00|  18|2024-03-04 00:00:00|             4200.0|
|2024-03-05 18:00:00|  18|2024-03-05 00:00:00|             5640.0|
|2024-03-05 18:00:00|  18|2024-03-05 00:00:00|             5640.0|
+-------------------+----+-------------------+-------------------+



### Exercises

**Q6 · `etl(customers)`** How many days had each customer been registered on 2024-04-01? Show name, signup_date and days_registered, sorted by signup_date.

Expected output:

```
+----------------+-----------+---------------+
|            name|signup_date|days_registered|
+----------------+-----------+---------------+
|    Aziz Karimov| 2024-01-05|             87|
|   Bella Schmidt| 2024-01-20|             72|
|     Carlos Ruiz| 2024-02-02|             59|
|Dilnoza Rahimova| 2024-02-14|             47|
|     Emma Wilson| 2024-03-01|             31|
|    Farid Aliyev| 2024-03-10|             22|
|       Grace Lee| 2024-03-15|             17|
+----------------+-----------+---------------+
```

In [ ]:
def etl(customers):
    # Write code here: return a DataFrame
    return customers

etl(customers).show()

**Q7 · `etl(signups_raw)`** The sign-up dates come in three formats (`2024-01-05`, `20/01/2024`, `2024/02/02`). Turn them into one date column `signup_day`. Show signup_id, signup_date and signup_day, sorted by signup_id.

Expected output:

```
+---------+-----------+----------+
|signup_id|signup_date|signup_day|
+---------+-----------+----------+
|        1| 2024-01-05|2024-01-05|
|        2| 20/01/2024|2024-01-20|
|        3| 2024/02/02|2024-02-02|
|        4| 14/02/2024|2024-02-14|
|        5| 2024-03-01|2024-03-01|
|        6| 2024-03-02|2024-03-02|
+---------+-----------+----------+
```

In [ ]:
def etl(signups_raw):
    # Write code here: return a DataFrame
    return signups_raw

etl(signups_raw).show()

**Q8 · `etl(events)`** Show the evening events (18:00 or later), duplicates removed: customer_id, event_time and page, sorted by event_time.

Expected output:

```
+-----------+-------------------+-------+
|customer_id|         event_time|   page|
+-----------+-------------------+-------+
|          2|2024-03-01 20:00:00|   home|
|          2|2024-03-01 20:20:00|product|
|          2|2024-03-01 20:45:00|   cart|
|          2|2024-03-04 18:00:00|   home|
|          2|2024-03-05 18:00:00|   home|
+-----------+-------------------+-------+
```

In [ ]:
def etl(events):
    # Write code here: return a DataFrame
    return events

etl(events).show()

## Lesson 4 · Aggregations: `groupBy(...).agg(...)`

`groupBy` collects the rows that have the same key, and `agg` computes values for each group. The result has **one row per group**: the key columns plus one column per aggregate.

```python
orders.groupBy("customer_id").agg(
    F.count("*").alias("orders"),
    F.sum("quantity").alias("items"),
)
```

Always `alias` an aggregate; otherwise the column is called something like `sum(quantity)`.

| Function | Gives |
|---|---|
| `F.count("*")` | the number of rows in the group |
| `F.count("col")` | the number of rows where col is **not null** |
| `F.countDistinct("col")` (also `F.count_distinct`) | the number of different non-null values |
| `F.sum`, `F.avg` (also `F.mean`), `F.min`, `F.max` | they **ignore nulls**: the avg of 1, null, 3 is 2 |
| `F.round(F.avg("x"), 2)` | round the result, not the inputs |
| `F.collect_list("x")`, `F.collect_set("x")` | all values / the different values, as an array (Lesson 8) |
| `F.first("x")` | some value of the group; which one isn't defined without an order |
| `F.min_by("x", "y")`, `F.max_by("x", "y")` | x from the row with the smallest / largest y |
| `F.stddev`, `F.variance` | spread (the sample versions) |

Ways to group:
* `groupBy("a", "b")` makes one group per (a, b) combination.
* `groupBy(F.month("order_date").alias("month"))` groups by an expression.
* `df.agg(...)` without `groupBy` gives **one row** for the whole table.
* `groupBy("x").count()` is a shortcut that adds a column named `count`.
* A null key forms its own group: the guest order shows up with customer_id null.

**Filter on a group (SQL `HAVING`).** `filter` after `agg`, on the alias: `.agg(F.count("*").alias("n")).filter(F.col("n") >= 3)`.

**Conditional aggregation** counts or sums only some rows of each group: `F.sum(F.when(cond, 1).otherwise(0))` or `F.count(F.when(cond, True))`. The second works because `count` skips the nulls that a `when` without `otherwise` produces. The same idea gives conditional sums: `F.sum(F.when(F.col("status") == "delivered", F.col("quantity")))`.

**One number from the whole table.** `total = orders.agg(F.sum("quantity")).first()[0]` gives a plain Python value that you can use in `F.lit(total)` or in a filter. Zillacode's #32 compares each ride with the overall average this way. You can stay inside Spark instead with a `crossJoin` of the one-row aggregate (Day 14), or with a window over the whole table (Day 14).

On Zillacode, aggregations are in #5, #8, #19, #20, #23, #25, #35, #36, #39 and #44. [GroupedData docs](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.GroupedData.agg.html).

In [ ]:
products.groupBy("category").agg(
    F.count("*").alias("products"),
    F.round(F.avg("price"), 2).alias("avg_price"),
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
).orderBy("category").show()

+-----------+--------+---------+---------+---------+
|   category|products|avg_price|min_price|max_price|
+-----------+--------+---------+---------+---------+
|      Books|       2|     20.0|     15.0|     25.0|
|Electronics|       2|    675.0|    150.0|   1200.0|
|       Home|       1|     40.0|     40.0|     40.0|
|    Kitchen|       2|     72.5|     60.0|     85.0|
+-----------+--------+---------+---------+---------+



In [ ]:
# count(*) vs count(col) vs countDistinct; the null key is its own group
orders.groupBy("customer_id").agg(
    F.count("*").alias("rows"),
    F.countDistinct("product_id").alias("different_products"),
    F.sum("quantity").alias("items"),
).orderBy("customer_id").show()
customers.agg(F.count("*").alias("rows"), F.count("email").alias("with_email"), F.countDistinct("city").alias("cities")).show()

+-----------+----+------------------+-----+
|customer_id|rows|different_products|items|
+-----------+----+------------------+-----+
|       NULL|   1|                 1|    1|
|          1|   4|                 4|    6|
|          2|   3|                 3|    4|
|          3|   3|                 3|    6|
|          4|   2|                 2|    4|
|          5|   2|                 2|    6|
|          6|   1|                 1|    2|
+-----------+----+------------------+-----+

+----+----------+------+
|rows|with_email|cities|
+----+----------+------+
|   7|         6|     4|
+----+----------+------+



In [ ]:
# HAVING: keep only the big groups
orders.groupBy("status").agg(F.count("*").alias("orders"), F.sum("quantity").alias("items")).filter(F.col("orders") >= 4).orderBy("status").show()
# Conditional aggregation: count some rows of each group
orders.groupBy(F.month("order_date").alias("month")).agg(
    F.count("*").alias("orders"),
    F.sum(F.when(F.col("status") == "delivered", 1).otherwise(0)).alias("delivered"),
    F.count(F.when(F.col("quantity") >= 2, True)).alias("multi_item"),
    F.sum(F.when(F.col("status") == "delivered", F.col("quantity"))).alias("delivered_items"),
).orderBy("month").show()

+---------+------+-----+
|   status|orders|items|
+---------+------+-----+
|delivered|    10|   20|
|  shipped|     4|    6|
+---------+------+-----+

+-----+------+---------+----------+---------------+
|month|orders|delivered|multi_item|delivered_items|
+-----+------+---------+----------+---------------+
|    1|     3|        3|         2|              6|
|    2|     4|        2|         1|              2|
|    3|     9|        5|         4|             12|
+-----+------+---------+----------+---------------+



In [ ]:
# One value from the whole table, used in a filter
avg_price = products.agg(F.avg("price")).first()[0]
print("average price:", avg_price)
products.filter(F.col("price") > avg_price).select("product_name", "price").show()

average price: 225.0
+------------+------+
|product_name| price|
+------------+------+
|      Laptop|1200.0|
+------------+------+



### Exercises

**Q9 · `etl(orders)`** Which customers placed 3 or more orders? Show customer_id and orders, sorted by customer_id.

Expected output:

```
+-----------+------+
|customer_id|orders|
+-----------+------+
|          1|     4|
|          2|     3|
|          3|     3|
+-----------+------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q10 · `etl(orders)`** For each customer who placed orders, count their delivered and their cancelled orders in two columns. Show customer_id, delivered and cancelled, sorted by customer_id. Leave out the guest order.

Expected output:

```
+-----------+---------+---------+
|customer_id|delivered|cancelled|
+-----------+---------+---------+
|          1|        3|        0|
|          2|        3|        0|
|          3|        2|        1|
|          4|        0|        1|
|          5|        1|        0|
|          6|        0|        0|
+-----------+---------+---------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q11 · `etl(orders)`** Which customers order more items per order, on average, than the average order? Show customer_id and avg_quantity (2 decimals), sorted by customer_id. The guest isn't a customer, but the guest order counts in the overall average.

Expected output:

```
+-----------+------------+
|customer_id|avg_quantity|
+-----------+------------+
|          3|         2.0|
|          4|         2.0|
|          5|         3.0|
|          6|         2.0|
+-----------+------------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

## Lesson 5 · 🔬 How Spark runs a `groupBy`

The rows of one key are spread over many partitions, so a `groupBy(...).agg(...)` runs in **two steps around a shuffle** (theory Day 3):

```
partition 1 rows ─► HashAggregate (partial_sum, partial_count) ─► one row per key seen here ─┐
partition 2 rows ─► HashAggregate (partial_sum, partial_count) ─► one row per key seen here ─┼─► Exchange hashpartitioning(key, 200) ─► HashAggregate (sum, count) ─► one row per key
partition 3 rows ─► HashAggregate (partial_sum, partial_count) ─► one row per key seen here ─┘
```

1. **Partial aggregation** before the shuffle: each task aggregates its own rows into one row per key it has seen. `avg` keeps a (sum, count) pair, so it can be finished later.
2. **The shuffle** (`Exchange hashpartitioning`): these partial rows, not the original rows, are sent so that each key lands in one partition (the key's hash picks the partition).
3. **Final aggregation**: each task merges the partial results of its keys.

So a billion rows with a thousand keys shuffle at most a thousand rows per task. It also makes a `groupBy` on a very frequent key harmless (on Day 14, the same key will slow down a join).

**One shuffle per `groupBy`, whatever the number of aggregates.** Put every aggregate of one key into **one** `agg(...)`, with conditional aggregation where needed (Lesson 4). Two `groupBy`s on the same key joined back together shuffle twice and add a join (Q13).

**Not every aggregate is that cheap:**

| You write | Plan | Cost |
|---|---|---|
| `sum`, `count`, `avg`, `min`/`max` of numbers or dates | `HashAggregate` | cheap: small fixed-size buffers in Tungsten's memory |
| `collect_list`, `collect_set`, `percentile` | `ObjectHashAggregate` | buffers grow with the group: one huge group means one huge list in one task |
| `max`/`min`/`first` of **text**, `dropDuplicates(["col"])` | `SortAggregate`, with a `Sort` before each step | a text value has no fixed size, so the hash buffers can't hold it: every partition is sorted first, which is slower |
| `F.countDistinct("x")` | **two** shuffles: first by (key, x) to remove duplicate pairs, then by key to count them | every distinct (key, x) pair travels. Two distinct counts add an `Expand` that copies each row once per counted column |
| `F.approx_count_distinct("x")` | one shuffle, with a small HyperLogLog sketch per group | an error of a few percent (its `rsd` parameter, 0.05 by default): far cheaper on big data when an estimate is enough |
| `df.agg(...)` without `groupBy` | `Exchange SinglePartition` | the final step runs as **one** task: fine for a few aggregates, dangerous for a sort or a window |
| `df.distinct()` | `HashAggregate` grouped by **every** column | a shuffle of the whole table |

**How many tasks after the shuffle?** `spark.sql.shuffle.partitions` (200 by default) sets how many partitions a shuffle writes. With AQE (on by default since Spark 3.2), once the map side has finished, Spark knows the real size of every partition and **coalesces** the small ones, aiming at about 64 MB each (`spark.sql.adaptive.advisoryPartitionSizeInBytes`). That's why Lesson 1's `groupBy` ran one task after its shuffle instead of 200. [AQE in the performance tuning guide](https://spark.apache.org/docs/latest/sql-performance-tuning.html#adaptive-query-execution).

In [ ]:
# An exact distinct count: two shuffles. The first one removes duplicate (customer_id, product_id) pairs
orders.groupBy("customer_id").agg(F.countDistinct("product_id").alias("products")).explain()
# An approximate one: a single shuffle, with a small sketch per group
orders.groupBy("customer_id").agg(F.approx_count_distinct("product_id").alias("products")).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[customer_id#56], functions=[count(distinct product_id#57)])
   +- Exchange hashpartitioning(customer_id#56, 200), ENSURE_REQUIREMENTS, [plan_id=1471]
      +- HashAggregate(keys=[customer_id#56], functions=[partial_count(distinct product_id#57)])
         +- HashAggregate(keys=[customer_id#56, product_id#57], functions=[])
            +- Exchange hashpartitioning(customer_id#56, product_id#57, 200), ENSURE_REQUIREMENTS, [plan_id=1467]
               +- HashAggregate(keys=[customer_id#56, product_id#57], functions=[])
                  +- FileScan parquet [customer_id#56,product_id#57] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/shop/orders], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<customer_id:int,product_id:int>


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[customer_id#56], function

In [ ]:
# max() of a text column can't use HashAggregate's fixed-size buffers: Spark sorts and uses SortAggregate
orders.groupBy("customer_id").agg(F.max("status").alias("last_status")).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortAggregate(key=[customer_id#56], functions=[max(status#60)])
   +- Sort [customer_id#56 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(customer_id#56, 200), ENSURE_REQUIREMENTS, [plan_id=1500]
         +- SortAggregate(key=[customer_id#56], functions=[partial_max(status#60)])
            +- Sort [customer_id#56 ASC NULLS FIRST], false, 0
               +- FileScan parquet [customer_id#56,status#60] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/shop/orders], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<customer_id:int,status:string>




In [ ]:
# Two groupBys on the same key, joined back: two shuffles, and a join on top
items = orders.groupBy("customer_id").agg(F.sum("quantity").alias("items"))
n_orders = orders.groupBy("customer_id").agg(F.count("*").alias("orders"))
items.join(n_orders, "customer_id").explain()
# One groupBy with both aggregates: one shuffle
orders.groupBy("customer_id").agg(F.sum("quantity").alias("items"), F.count("*").alias("orders")).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [customer_id#56, items#951L, orders#959L]
   +- BroadcastHashJoin [customer_id#56], [customer_id#968], Inner, BuildRight, false, false
      :- HashAggregate(keys=[customer_id#56], functions=[sum(quantity#58)])
      :  +- Exchange hashpartitioning(customer_id#56, 200), ENSURE_REQUIREMENTS, [plan_id=1547]
      :     +- HashAggregate(keys=[customer_id#56], functions=[partial_sum(quantity#58)])
      :        +- Filter isnotnull(customer_id#56)
      :           +- FileScan parquet [customer_id#56,quantity#58] Batched: true, DataFilters: [isnotnull(customer_id#56)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/you/spark-lab/data/shop/orders], PartitionFilters: [], PushedFilters: [IsNotNull(customer_id)], ReadSchema: struct<customer_id:int,quantity:int>
      +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, true] as bigint)),false), [plan_id=1552]
         +- HashAggregate(keys

### Exercises

**Q12 · 🔍 Read the plan** How many shuffles does this query need? What does the `Expand` operator do, and why does it make the data bigger before the first shuffle?

In [ ]:
q12 = orders.groupBy(F.month("order_date").alias("month")).agg(
    F.countDistinct("customer_id").alias("customers"),
    F.countDistinct("product_id").alias("products"))
q12.explain()
# Your answer:

**Q13 · ⚡ `etl(orders)`** For each customer (leave out the guest order), show orders (all their orders) and delivered_items (the quantity of their delivered orders, 0 if none), sorted by customer_id. `slow_etl` in the cell gets it right with two `groupBy`s and a join; write it with one `groupBy`.

Expected output:

```
+-----------+------+---------------+
|customer_id|orders|delivered_items|
+-----------+------+---------------+
|          1|     4|              5|
|          2|     3|              4|
|          3|     3|              5|
|          4|     2|              0|
|          5|     2|              5|
|          6|     1|              0|
+-----------+------+---------------+
```

**Target:** no join, and only two `Exchange` operators: `hashpartitioning(customer_id, 200)` for the `groupBy` and `rangepartitioning` for the `orderBy`. Compare with the plan of `slow_etl`.

In [ ]:
# Correct, but slow: two groupBys on the same key, joined back together
def slow_etl(orders):
    known = orders.filter(F.col("customer_id").isNotNull())
    counts = known.groupBy("customer_id").agg(F.count("*").alias("orders"))
    delivered = (known.filter(F.col("status") == "delivered")
                      .groupBy("customer_id").agg(F.sum("quantity").alias("delivered_items")))
    return (counts.join(delivered, "customer_id", "left")
                  .select("customer_id", "orders", F.coalesce("delivered_items", F.lit(0)).alias("delivered_items"))
                  .orderBy("customer_id"))

def etl(orders):
    # Write code here: the same rows with one groupBy
    return slow_etl(orders)

etl(orders).show()
etl(orders).explain()

## Lesson 6 · Aggregate first, then join: the fan-out trap

The Laptop has 3 orders and 2 reviews. Join products to orders and then to reviews, and the Laptop gets 3 × 2 = **6 rows**: every order repeats once per review. A `sum` or `count` over those rows counts everything several times. Nothing fails; the numbers are just wrong.

**The fix, used by many Zillacode problems (#10, #20, #23):**
1. Aggregate each detail table by the key **first**, so you have one row per key: `items per product`, `reviews per product`.
2. Join these small results to the main table with **left** joins, so keys without details stay in the answer.
3. Turn the nulls of the keys without details into 0 with `F.coalesce(col, F.lit(0))`, if the task wants 0.

The rule of thumb: never join two "many" tables to the same key before aggregating them.

🔬 **In the plan.** The wrong version is also the more expensive one: the joins multiply the rows *before* the `groupBy` shuffles them, so more rows travel. Aggregating first shrinks each detail table to one row per key before any join.

In [ ]:
# Wrong: two detail tables on the same key multiply each other's rows
wrong = (products.join(orders, "product_id", "left")
                 .join(reviews, "product_id", "left")
                 .groupBy("product_id", "product_name")
                 .agg(F.sum("quantity").alias("items"), F.count("review_id").alias("reviews")))
wrong.orderBy("product_id").show()

+----------+------------+-----+-------+
|product_id|product_name|items|reviews|
+----------+------------+-----+-------+
|       101|      Laptop|    6|      6|
|       102|  Headphones|    3|      2|
|       103|Coffee Maker|    8|      6|
|       104|     Blender|    3|      0|
|       105|       Novel|   24|      6|
|       106|    Cookbook|    3|      2|
|       107|   Desk Lamp| NULL|      0|
+----------+------------+-----+-------+



In [ ]:
# Right: one row per product in each summary, then left joins, then 0 for "no rows"
items = orders.groupBy("product_id").agg(F.sum("quantity").alias("items"))
review_counts = reviews.groupBy("product_id").agg(F.count("*").alias("reviews"))
right = (products.join(items, "product_id", "left")
                 .join(review_counts, "product_id", "left")
                 .select("product_id", "product_name",
                         F.coalesce("items", F.lit(0)).alias("items"),
                         F.coalesce("reviews", F.lit(0)).alias("reviews")))
right.orderBy("product_id").show()

+----------+------------+-----+-------+
|product_id|product_name|items|reviews|
+----------+------------+-----+-------+
|       101|      Laptop|    3|      2|
|       102|  Headphones|    3|      1|
|       103|Coffee Maker|    4|      2|
|       104|     Blender|    3|      0|
|       105|       Novel|   12|      2|
|       106|    Cookbook|    3|      1|
|       107|   Desk Lamp|    0|      0|
+----------+------------+-----+-------+



### Exercise

**Q14 · `etl(customers, orders, reviews)`** For every customer, show name, orders (the number of orders) and reviews (the number of reviews written), both 0 when there are none. Sort by customer_id.

Expected output:

```
+----------------+------+-------+
|            name|orders|reviews|
+----------------+------+-------+
|    Aziz Karimov|     4|      2|
|   Bella Schmidt|     3|      2|
|     Carlos Ruiz|     3|      2|
|Dilnoza Rahimova|     2|      0|
|     Emma Wilson|     2|      1|
|    Farid Aliyev|     1|      1|
|       Grace Lee|     0|      0|
+----------------+------+-------+
```

In [ ]:
def etl(customers, orders, reviews):
    # Write code here: return a DataFrame
    return customers

etl(customers, orders, reviews).show()

## Lesson 7 · Pivot: rows into columns

`df.groupBy(row_key).pivot(column, [values]).agg(expression)` gives one output row per `row_key` and **one column per value** of `column`, holding the aggregate for that combination.

* **Pass the list of values** when you know them. The columns then come out in that order, and a value with no rows still gets its column. Without the list, Spark first runs an extra job to find the values, and sorts them.
* A combination without rows is **null**. Use `.na.fill(0, subset=[the pivot columns])` or `coalesce` if the task wants 0. A plain `.na.fill(0)` fills **every** number column, including a null key column.
* The new columns are **named after the values** (`Books`, `cancelled`). Rename them with `withColumnRenamed` if the output schema wants other names.
* With a single aggregate, the shortcuts `.count()` and `.sum("x")` work after `pivot`.
* The reverse, columns into rows, is `df.unpivot(ids, value_columns, "name_column", "value_column")` (Spark 3.4 and later), or `F.expr("stack(2, 'jan', jan, 'feb', feb)")` on older versions.

On Zillacode, #3 pivots rents by property type and then adds the columns up. [pivot docs](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.GroupedData.pivot.html).

🔬 **In the plan.** A pivot runs as two aggregations: first by (row key, pivot value), then by the row key, where a `pivotfirst` function puts each value into its column. Without a value list, Spark must find the values first: **as soon as you call `.pivot("category")`**, before any action, it runs a whole extra query (a `distinct` of the column, with the join it depends on). The next cell counts the jobs: building and running the pivot without values takes more than twice as many.

In [ ]:
# Without a value list: the values found in the data, sorted; empty combinations are null
(orders.join(products, "product_id")
       .groupBy("customer_id")
       .pivot("category")
       .agg(F.sum("quantity"))
       .orderBy("customer_id")
       .show())

+-----------+-----+-----------+-------+
|customer_id|Books|Electronics|Kitchen|
+-----------+-----+-----------+-------+
|       NULL| NULL|       NULL|      1|
|          1|    3|          2|      1|
|          2|    1|          3|   NULL|
|          3|    4|       NULL|      1|
|          4|    2|       NULL|      2|
|          5|    5|          1|   NULL|
|          6| NULL|       NULL|      2|
+-----------+-----+-----------+-------+



In [ ]:
# With a value list (Home has no orders, but gets a column) and 0 instead of null.
# subset= limits the fill to the pivot columns: a plain na.fill(0) would also turn the guest's null customer_id into 0.
categories = ["Books", "Electronics", "Kitchen", "Home"]
(orders.join(products, "product_id")
       .groupBy("customer_id")
       .pivot("category", categories)
       .agg(F.sum("quantity"))
       .na.fill(0, subset=categories)
       .orderBy("customer_id")
       .show())

+-----------+-----+-----------+-------+----+
|customer_id|Books|Electronics|Kitchen|Home|
+-----------+-----+-----------+-------+----+
|       NULL|    0|          0|      1|   0|
|          1|    3|          2|      1|   0|
|          2|    1|          3|      0|   0|
|          3|    4|          0|      1|   0|
|          4|    2|          0|      2|   0|
|          5|    5|          1|      0|   0|
|          6|    0|          0|      2|   0|
+-----------+-----+-----------+-------+----+



In [ ]:
# The reverse: columns into rows
wide = spark.createDataFrame([(1, 10, 20), (2, 30, None)], "store int, jan int, feb int")
wide.show()
wide.unpivot("store", ["jan", "feb"], "month", "sales").show()

+-----+---+----+
|store|jan| feb|
+-----+---+----+
|    1| 10|  20|
|    2| 30|NULL|
+-----+---+----+

+-----+-----+-----+
|store|month|sales|
+-----+-----+-----+
|    1|  jan|   10|
|    1|  feb|   20|
|    2|  jan|   30|
|    2|  feb| NULL|
+-----+-----+-----+



In [ ]:
# Without a value list, building the DataFrame already runs a query to find the categories
joined = orders.join(products, "product_id")
measure(lambda: joined.groupBy("customer_id").pivot("category").agg(F.sum("quantity")).collect(),
        "pivot without values")
measure(lambda: joined.groupBy("customer_id").pivot("category", ["Books", "Electronics", "Home", "Kitchen"])
                     .agg(F.sum("quantity")).collect(),
        "pivot with values")

pivot without values: 0.20 s | 9 jobs, 9 stages, 9 tasks | read 17.4 KB | shuffle 1.9 KB | spill 0 B
pivot with values: 0.10 s | 4 jobs, 4 stages, 4 tasks | read 8.7 KB | shuffle 1.5 KB | spill 0 B


### Exercise

**Q15 · `etl(orders)`** Make a table with one row per month and one column per status, holding the number of orders, with 0 where there are none. Columns: month, cancelled, delivered, shipped. Sort by month.

Expected output:

```
+-----+---------+---------+-------+
|month|cancelled|delivered|shipped|
+-----+---------+---------+-------+
|    1|        0|        3|      0|
|    2|        1|        2|      1|
|    3|        1|        5|      3|
+-----+---------+---------+-------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

## Lesson 8 · Arrays: `split`, `explode` and `collect_list`

An array column holds a list in every row. You get one from `F.split` (Day 12), or from `F.collect_list` / `F.collect_set` in an `agg`.

| Function | What it does |
|---|---|
| `F.explode(arr)` | one row per item. Rows whose array is null or empty **disappear**. |
| `F.explode_outer(arr)` | the same, but those rows stay, with a null item |
| `F.posexplode(arr)` | also gives each item's position (from 0) |
| `F.size(arr)`, `F.array_contains(arr, v)`, `F.element_at(arr, i)` | count the items, test for one, pick one (from 1) |
| `F.array_distinct`, `F.sort_array`, `F.array_join(arr, ", ")` | remove repeats, sort, turn back into text |
| `F.collect_list(x)`, `F.collect_set(x)` inside `agg` | the reverse of explode: many rows into one array (`set` without repeats) |

The typical job is a column with a list inside a text (`"fast delivery;good price"`): `split` it, `explode` it, then `groupBy` the items to count them.

`collect_list` and `collect_set` keep **no particular order**. When the output must be stable (and a checker compares it), wrap them in `F.sort_array(...)`.

[explode docs](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.explode.html).

🔬 **In the plan.** `explode` is the `Generate` operator. It runs inside a stage, before any shuffle, and it multiplies the rows: filter and select what you need **before** exploding, so fewer rows are copied and shuffled. `collect_list` and `collect_set` run as `ObjectHashAggregate` (Lesson 5): each group's whole list is built in one task's memory.

In [ ]:
# One row per tag; review 6 (no tags) disappears with explode, stays with explode_outer
reviews.select("review_id", F.explode(F.split("tags", ";")).alias("tag")).show()
reviews.select("review_id", F.explode_outer(F.split("tags", ";")).alias("tag")).filter(F.col("review_id") == 6).show()

+---------+-------------+
|review_id|          tag|
+---------+-------------+
|        1|fast delivery|
|        1|   good price|
|        2|   good price|
|        3|slow delivery|
|        4|       broken|
|        4|slow delivery|
|        5|fast delivery|
|        7|   good price|
|        7|         gift|
|        8|         gift|
+---------+-------------+

+---------+----+
|review_id| tag|
+---------+----+
|        6|NULL|
+---------+----+



In [ ]:
reviews.select(
    "review_id",
    F.split("tags", ";").alias("tag_list"),
    F.array_contains(F.split("tags", ";"), "gift").alias("is_gift"),
).show(truncate=False)

+---------+---------------------------+-------+
|review_id|tag_list                   |is_gift|
+---------+---------------------------+-------+
|1        |[fast delivery, good price]|false  |
|2        |[good price]               |false  |
|3        |[slow delivery]            |false  |
|4        |[broken, slow delivery]    |false  |
|5        |[fast delivery]            |false  |
|6        |NULL                       |NULL   |
|7        |[good price, gift]         |true   |
|8        |[gift]                     |true   |
+---------+---------------------------+-------+



In [ ]:
# Many rows into one array per group, sorted so the result is stable
orders.filter(F.col("customer_id").isNotNull()).groupBy("customer_id").agg(
    F.sort_array(F.collect_list("product_id")).alias("products"),
    F.sort_array(F.collect_set("status")).alias("statuses"),
    F.array_join(F.sort_array(F.collect_set("status")), ", ").alias("statuses_text"),
).orderBy("customer_id").show(truncate=False)

+-----------+--------------------+----------------------+--------------------+
|customer_id|products            |statuses              |statuses_text       |
+-----------+--------------------+----------------------+--------------------+
|1          |[101, 102, 103, 105]|[delivered, shipped]  |delivered, shipped  |
|2          |[101, 102, 106]     |[delivered]           |delivered           |
|3          |[103, 105, 108]     |[cancelled, delivered]|cancelled, delivered|
|4          |[104, 106]          |[cancelled, shipped]  |cancelled, shipped  |
|5          |[101, 105]          |[delivered, shipped]  |delivered, shipped  |
|6          |[103]               |[shipped]             |shipped             |
+-----------+--------------------+----------------------+--------------------+



### Exercises

**Q16 · `etl(reviews)`** Which tags are used most? Show tag and reviews (the number of reviews with that tag), most first; on a tie, by tag from A to Z.

Expected output:

```
+-------------+-------+
|          tag|reviews|
+-------------+-------+
|   good price|      3|
|fast delivery|      2|
|         gift|      2|
|slow delivery|      2|
|       broken|      1|
+-------------+-------+
```

In [ ]:
def etl(reviews):
    # Write code here: return a DataFrame
    return reviews

etl(reviews).show()

**Q17 · `etl(orders, products)`** For each customer, list the different categories they ordered from, from A to Z, as one text separated by `, `. Show customer_id and categories, sorted by customer_id. Leave out the guest; use an inner join.

Expected output:

```
+-----------+---------------------------+
|customer_id|categories                 |
+-----------+---------------------------+
|1          |Books, Electronics, Kitchen|
|2          |Books, Electronics         |
|3          |Books, Kitchen             |
|4          |Books, Kitchen             |
|5          |Books, Electronics         |
|6          |Kitchen                    |
+-----------+---------------------------+
```

In [ ]:
def etl(orders, products):
    # Write code here: return a DataFrame
    return orders

etl(orders, products).show(truncate=False)

## Mixed problems
Several lessons in one problem, like on Zillacode. About 10 to 15 minutes each.

**Q18 · `etl(orders)`** A monthly report: month (text like `2024-01`), orders, customers (different customers; the guest doesn't count), delivered_items (the quantity of delivered orders) and cancelled_pct (the percent of the month's orders that were cancelled, 1 decimal). Sort by month.

Expected output:

```
+-------+------+---------+---------------+-------------+
|  month|orders|customers|delivered_items|cancelled_pct|
+-------+------+---------+---------------+-------------+
|2024-01|     3|        2|              6|          0.0|
|2024-02|     4|        4|              2|         25.0|
|2024-03|     9|        6|             12|         11.1|
+-------+------+---------+---------------+-------------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q19 · `etl(customers, orders)`** How long did each customer take to place a first order? Show customer_id, signup_date, first_order_date and days_to_first_order. Customers without orders show nulls. Sort by customer_id.

Expected output:

```
+-----------+-----------+----------------+-------------------+
|customer_id|signup_date|first_order_date|days_to_first_order|
+-----------+-----------+----------------+-------------------+
|          1| 2024-01-05|      2024-01-10|                  5|
|          2| 2024-01-20|      2024-01-25|                  5|
|          3| 2024-02-02|      2024-02-03|                  1|
|          4| 2024-02-14|      2024-02-15|                  1|
|          5| 2024-03-01|      2024-03-02|                  1|
|          6| 2024-03-10|      2024-03-12|                  2|
|          7| 2024-03-15|            NULL|               NULL|
+-----------+-----------+----------------+-------------------+
```

In [ ]:
def etl(customers, orders):
    # Write code here: return a DataFrame
    return customers

etl(customers, orders).show()

**Q20 · ⚡ `etl(orders, products)`** A category report for March 2024: category, items (the quantity of all orders) and buyers (the number of different customers; the guest doesn't count), sorted by category. Use an inner join with products. `slow_etl` gets it right, but its plan shows two problems. Find them, then write a version without them.

Expected output:

```
+-----------+-----+------+
|   category|items|buyers|
+-----------+-----+------+
|      Books|   11|     3|
|Electronics|    2|     2|
|    Kitchen|    4|     2|
+-----------+-----+------+
```

**Target:** the date range is in `PushedFilters`, and each table is scanned once: one `FileScan` of orders and one of products in the plan (`slow_etl` has two of each).

In [ ]:
# Correct, but wasteful
def slow_etl(orders, products):
    march = orders.filter(F.date_format("order_date", "yyyy-MM") == "2024-03")
    joined = march.join(products, "product_id")
    items = joined.groupBy("category").agg(F.sum("quantity").alias("items"))
    buyers = joined.groupBy("category").agg(F.countDistinct("customer_id").alias("buyers"))
    return items.join(buyers, "category").orderBy("category")

def etl(orders, products):
    # Write code here: return a DataFrame
    return slow_etl(orders, products)

etl(orders, products).show()
etl(orders, products).explain()

## Answers
Try each exercise yourself first. Click an exercise to open its answer.

<details>
<summary><b>Q1</b> &nbsp;How many Exchange operators does this query have, and what does each one do? How many stages will it run?</summary>

Two `Exchange`s. `hashpartitioning(status, 200)` brings each status's partial sums together for the `groupBy`; `rangepartitioning(items DESC NULLS LAST, 200)` splits the totals into sorted ranges for the `orderBy`, and the `Sort` above it sorts each range, so the result is sorted as a whole.

`measure` shows **4 stages**, one more than two shuffles suggest: (1) read the file and compute the partial sums; (2) a small job that **samples** the totals to choose the range boundaries (that's how `orderBy` works, theory Day 3), which computes the final sums once just for the sample; (3) the final sums again, written into the range shuffle; (4) the sort. Each stage ran one task because AQE coalesced the 200 partitions of this tiny data. A global sort is expensive: sort only when the output needs it, and prefer `orderBy(...).limit(n)` (no full shuffle) when you want the top rows.

</details>

<details>
<summary><b>Q2</b> &nbsp;Which columns does Spark read from each table for this query? Where is the status condition checked?</summary>

customers: `customer_id` and `name` (its `ReadSchema`); orders: `customer_id`, `order_date` and `status`. Only the join key and the selected columns are read, plus `status` for the filter.

The status condition is checked in the orders branch, **before** the join: in the `Filter` right above the orders scan, and in that scan's `PushedFilters` (`EqualTo(status,cancelled)`), so the Parquet reader gets it too. Both tables are tiny, so Spark broadcasts one side (`BroadcastHashJoin`, here the filtered orders: `BuildRight`); Day 14 explains how it chooses.

</details>

<details>
<summary><b>Q3</b> &nbsp;Predict: how many shuffles does orders.select("status").distinct().count() need, and why?</summary>

Two shuffles, so three stages. `distinct()` is a `groupBy` on every selected column (here `status`): partial `HashAggregate`, `Exchange hashpartitioning(status, 200)`, final `HashAggregate`. `count()` on that result is an aggregation without a key: partial counts, `Exchange SinglePartition`, then the final count in one task. `measure` shows 3 jobs and 3 stages of one task each, and the query's diagram in the SQL / DataFrame tab shows both exchanges.

</details>

<details>
<summary><b>Q4</b> &nbsp;Which of the three conditions reach the Parquet reader, and which are only checked after reading?</summary>

Pushed (in `PushedFilters`): `GreaterThanOrEqual(signup_date,2024-02-01)` and `IsNotNull(city)`, plus `IsNotNull(signup_date)` and `IsNotNull(email)`, which Catalyst inferred because a null can't pass these conditions. Not pushed: `EndsWith(lower(email), .uz)`. `lower()` wraps the column, so only Spark's `Filter` checks it, after reading.

To push it, test the raw column, `F.col("email").endswith(".uz")`, when the emails are stored in lower case (clean them once, when the table is written, instead of in every query). Also look at `ReadSchema`: every column is read, because the query has no `select`.

</details>

<details>
<summary><b>Q5</b> &nbsp;Show the delivered orders placed in March 2024: order_id, customer_id and order_date, sorted by order_id ...</summary>

```python
def etl(orders):
    return (orders.filter((F.col("status") == "delivered")
                          & (F.col("order_date") >= "2024-03-01")
                          & (F.col("order_date") < "2024-04-01"))
                  .select("order_id", "customer_id", "order_date")
                  .orderBy("order_id"))

etl(orders).show()
```

`F.month("order_date") == 3` (with a year check) returns the same rows, but its plan pushes only `IsNotNull(order_date)`: the reader returns every row group and Spark filters afterwards. `F.col("order_date").between("2024-03-01", "2024-03-31")` is pushed too.

</details>

<details>
<summary><b>Q6</b> &nbsp;How many days had each customer been registered on 2024-04-01? Show name, signup_date and days_registered, ...</summary>

```python
def etl(customers):
    return (customers.withColumn("days_registered", F.datediff(F.to_date(F.lit("2024-04-01")), F.col("signup_date")))
                     .select("name", "signup_date", "days_registered")
                     .orderBy("signup_date"))

etl(customers).show()
```

</details>

<details>
<summary><b>Q7</b> &nbsp;The sign-up dates come in three formats (2024-01-05, 20/01/2024, 2024/02/02). Turn them into one date column ...</summary>

```python
def etl(signups_raw):
    return (signups_raw.select("signup_id", "signup_date",
                                F.coalesce(F.to_date("signup_date", "yyyy-MM-dd"),
                                           F.to_date("signup_date", "dd/MM/yyyy"),
                                           F.to_date("signup_date", "yyyy/MM/dd")).alias("signup_day"))
                       .orderBy("signup_id"))

etl(signups_raw).show()
```

Each `to_date` gives null when its pattern doesn't fit, and `coalesce` takes the first one that worked.

</details>

<details>
<summary><b>Q8</b> &nbsp;Show the evening events (18:00 or later), duplicates removed: customer_id, event_time and page, sorted by ...</summary>

```python
def etl(events):
    return (events.distinct()
                  .filter(F.hour("event_time") >= 18)
                  .orderBy("event_time"))

etl(events).show()
```

</details>

<details>
<summary><b>Q9</b> &nbsp;Which customers placed 3 or more orders? Show customer_id and orders, sorted by customer_id.</summary>

```python
def etl(orders):
    return (orders.groupBy("customer_id")
                  .agg(F.count("*").alias("orders"))
                  .filter(F.col("orders") >= 3)
                  .orderBy("customer_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q10</b> &nbsp;For each customer who placed orders, count their delivered and their cancelled orders in two columns. Show ...</summary>

```python
def etl(orders):
    return (orders.filter(F.col("customer_id").isNotNull())
                  .groupBy("customer_id")
                  .agg(F.sum(F.when(F.col("status") == "delivered", 1).otherwise(0)).alias("delivered"),
                       F.sum(F.when(F.col("status") == "cancelled", 1).otherwise(0)).alias("cancelled"))
                  .orderBy("customer_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q11</b> &nbsp;Which customers order more items per order, on average, than the average order? Show customer_id and ...</summary>

```python
def etl(orders):
    overall = orders.agg(F.avg("quantity")).first()[0]
    return (orders.filter(F.col("customer_id").isNotNull())
                  .groupBy("customer_id")
                  .agg(F.avg("quantity").alias("avg_quantity"))
                  .filter(F.col("avg_quantity") > overall)
                  .select("customer_id", F.round("avg_quantity", 2).alias("avg_quantity"))
                  .orderBy("customer_id"))

etl(orders).show()
```

Filter on the unrounded average, then round for the output: rounding first can move a value across the limit.

</details>

<details>
<summary><b>Q12</b> &nbsp;How many shuffles does a groupBy with two countDistinct need? What does Expand do?</summary>

Two shuffles. `Expand` copies every row once per distinct count: one copy with `product_id` and `gid = 1`, one with `customer_id` and `gid = 2`, the other column set to null. The `HashAggregate`s around the first `Exchange` (grouped by month, both columns and `gid`) remove the duplicates of both columns in one pass; the ones around the second `Exchange` count the values of each `gid` (`count(...) FILTER (WHERE gid = ...)`). So the first shuffle can move twice as many rows as the table has; with three distinct counts, three times. On big data, `approx_count_distinct` avoids all of it.

</details>

<details>
<summary><b>Q13</b> &nbsp;For each customer (leave out the guest order), show orders (all their orders) and delivered_items (the ...</summary>

```python
def etl(orders):
    return (orders.filter(F.col("customer_id").isNotNull())
                  .groupBy("customer_id")
                  .agg(F.count("*").alias("orders"),
                       F.sum(F.when(F.col("status") == "delivered", F.col("quantity")).otherwise(0))
                        .alias("delivered_items"))
                  .orderBy("customer_id"))

etl(orders).show()
```

`.otherwise(0)` makes customers without delivered orders get 0 instead of null, which replaces the `coalesce` of the slow version. The slow plan has three `Exchange`s (two `groupBy`s and the `orderBy`) plus a `BroadcastExchange` for the join, and it reads the file twice.

</details>

<details>
<summary><b>Q14</b> &nbsp;For every customer, show name, orders (the number of orders) and reviews (the number of reviews written), ...</summary>

```python
def etl(customers, orders, reviews):
    order_counts = orders.groupBy("customer_id").agg(F.count("*").alias("orders"))
    review_counts = reviews.groupBy("customer_id").agg(F.count("*").alias("reviews"))
    return (customers.join(order_counts, "customer_id", "left")
                     .join(review_counts, "customer_id", "left")
                     .orderBy("customer_id")
                     .select("name",
                             F.coalesce("orders", F.lit(0)).alias("orders"),
                             F.coalesce("reviews", F.lit(0)).alias("reviews")))

etl(customers, orders, reviews).show()
```

</details>

<details>
<summary><b>Q15</b> &nbsp;Make a table with one row per month and one column per status, holding the number of orders, with 0 where ...</summary>

```python
def etl(orders):
    return (orders.groupBy(F.month("order_date").alias("month"))
                  .pivot("status", ["cancelled", "delivered", "shipped"])
                  .count()
                  .na.fill(0)
                  .orderBy("month"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q16</b> &nbsp;Which tags are used most? Show tag and reviews (the number of reviews with that tag), most first; on a tie, ...</summary>

```python
def etl(reviews):
    return (reviews.select(F.explode(F.split("tags", ";")).alias("tag"))
                   .groupBy("tag")
                   .agg(F.count("*").alias("reviews"))
                   .orderBy(F.desc("reviews"), "tag"))

etl(reviews).show()
```

</details>

<details>
<summary><b>Q17</b> &nbsp;For each customer, list the different categories they ordered from, from A to Z, as one text separated by , . ...</summary>

```python
def etl(orders, products):
    return (orders.filter(F.col("customer_id").isNotNull())
                  .join(products, "product_id")
                  .groupBy("customer_id")
                  .agg(F.array_join(F.sort_array(F.collect_set("category")), ", ").alias("categories"))
                  .orderBy("customer_id"))

etl(orders, products).show(truncate=False)
```

</details>

<details>
<summary><b>Q18</b> &nbsp;A monthly report: month (text like 2024-01), orders, customers (different customers; the guest doesn't ...</summary>

```python
def etl(orders):
    return (orders.groupBy(F.date_format("order_date", "yyyy-MM").alias("month"))
                  .agg(F.count("*").alias("orders"),
                       F.countDistinct("customer_id").alias("customers"),
                       F.sum(F.when(F.col("status") == "delivered", F.col("quantity")).otherwise(0)).alias("delivered_items"),
                       F.round(F.avg(F.when(F.col("status") == "cancelled", 1).otherwise(0)) * 100, 1).alias("cancelled_pct"))
                  .orderBy("month"))

etl(orders).show()
```

The average of a 1/0 column is the share of rows where the condition is true: a handy trick for percentages.

</details>

<details>
<summary><b>Q19</b> &nbsp;How long did each customer take to place a first order? Show customer_id, signup_date, first_order_date and ...</summary>

```python
def etl(customers, orders):
    first_orders = orders.groupBy("customer_id").agg(F.min("order_date").alias("first_order_date"))
    return (customers.join(first_orders, "customer_id", "left")
                     .select("customer_id", "signup_date", "first_order_date",
                             F.datediff("first_order_date", "signup_date").alias("days_to_first_order"))
                     .orderBy("customer_id"))

etl(customers, orders).show()
```

Aggregate first (one row per customer), then a left join: Lesson 6's pattern.

</details>

<details>
<summary><b>Q20</b> &nbsp;A category report for March 2024: category, items (the quantity of all orders) and buyers (the number of ...</summary>

```python
def etl(orders, products):
    return (orders.filter((F.col("order_date") >= "2024-03-01") & (F.col("order_date") < "2024-04-01"))
                  .join(products, "product_id")
                  .groupBy("category")
                  .agg(F.sum("quantity").alias("items"),
                       F.countDistinct("customer_id").alias("buyers"))
                  .orderBy("category"))

etl(orders, products).show()
```

The two problems of `slow_etl`: (1) `date_format(...) == "2024-03"` wraps the column in a function, so its plan pushes only `IsNotNull(order_date)`; (2) the same `joined` DataFrame feeds two aggregations that are joined back, and a DataFrame is a recipe, not stored data: both branches read both files and join them again (two `FileScan`s of each table and two `BroadcastExchange`s), then a third join puts the results together. One `agg` with both aggregates fixes it. `countDistinct` ignores nulls, so the guest is left out without a filter.

</details>